# GQA 与 MLA（分组查询注意力 / 多头潜在注意力）

> 减少 KV Cache 显存的两种方案，LLaMA 3 用 GQA，DeepSeek-V2 用 MLA。

## 背景
标准 MHA 中每个 Q head 有独立的 KV head，KV Cache 随 head 数线性增长。
- GQA：多个 Q head 共享一组 KV head，减少 KV head 数
- MQA：所有 Q head 共享 1 组 KV（GQA 的极端情况）
- MLA：用低秩压缩 KV，缓存 latent 向量而非完整 KV

## 公式
**GQA**：将 n_heads 个 Q head 分成 n_groups 组，每组共享 1 对 KV head
$$\text{KV heads} = n\_groups, \quad \text{KV Cache} \propto n\_groups$$
**MLA**：KV = W_KV × c_KV，只缓存低秩 latent c_KV (d_c << d × n_heads)

## 复杂度
- GQA：KV Cache 从 O(n × h × d) 降到 O(n × h/g × d)，g=组大小
- MLA：KV Cache 从 O(n × h × d) 降到 O(n × d_c)，d_c=压缩维度
- 推理速度：GQA 简单有效，MLA 压缩率更高但实现复杂

## 考察点
- GQA 的组数选择：LLaMA 3 用 8 组（32 Q heads / 8 = 4 KV heads）
- MLA 的压缩维度：DeepSeek-V2 用 d_c=512，压缩约 4x
- 质量 vs 显存：MQA 质量下降明显，GQA 接近 MHA


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class LlamaGQA(nn.Module):
    def __init__(self, dim: int, num_query_heads: int, num_kv_heads: int, head_dim: int = None) -> None:
        super().__init__()
        self.h = num_query_heads
        self.g = num_kv_heads
        self.head_dim = head_dim if head_dim is not None else dim // num_query_heads
        assert num_query_heads % num_kv_heads == 0
        self.group = num_query_heads // num_kv_heads
        self.q_proj = nn.Linear(dim, num_query_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_query_heads * self.head_dim, dim, bias=False)
        self.scale = 1.0 / math.sqrt(self.head_dim)

    def forward(self, x: torch.Tensor, cache: torch.Tensor = None) -> torch.Tensor:
        B, N, _ = x.shape
        q = self.q_proj(x).view(B, N, self.h, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.g, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.g, self.head_dim).transpose(1, 2)
        if cache is not None:
            pk, pv = cache
            k = torch.cat([pk, k], dim=2); v = torch.cat([pv, v], dim=2)
        # 共享：把 g 个 KV 头扩到 h 个 Q 头
        k = k.repeat_interleave(self.group, dim=1)
        v = v.repeat_interleave(self.group, dim=1)
        attn = F.softmax((q @ k.transpose(-1, -2)) * self.scale, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, N, -1)
        return self.o_proj(out), (k, v)

In [ ]:
# MLA 简化教学版：KV 低秩压缩 + 上采样（省略 RoPE 分离通路）
class MLAAttention(nn.Module):
    def __init__(self, hidden_size: int = 4096, n_heads: int = 32, latent_kv_dim: int = 512) -> None:
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden_size // n_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        # 共享压缩：h -> 2*latent（K 与 V 的低维表示）
        self.kv_latent_proj = nn.Linear(hidden_size, 2 * latent_kv_dim, bias=False)
        # 上采样：latent -> 高维 K/V
        self.k_up_proj = nn.Linear(latent_kv_dim, hidden_size, bias=False)
        self.v_up_proj = nn.Linear(latent_kv_dim, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, past_kv: torch.Tensor = None) -> torch.Tensor:
        B, T, C = x.shape
        H, D = self.n_heads, self.head_dim
        q = self.q_proj(x).view(B, T, H, D).transpose(1, 2)          # [B,H,T,D]
        # 1. 压缩到低维 latent 并缓存（cache 只存 latent，省显存）
        k_c, v_c = self.kv_latent_proj(x).chunk(2, dim=-1)            # 各 [B,T,latent]
        if past_kv is not None:
            pk, pv = past_kv
            k_c = torch.cat([pk, k_c], dim=1); v_c = torch.cat([pv, v_c], dim=1)
        # 2. 上采样回高维再做 attention（训练时如此；推理可把 W_up 吸收进 W_q 优化）
        k = self.k_up_proj(k_c).view(B, -1, H, D).transpose(1, 2)
        v = self.v_up_proj(v_c).view(B, -1, H, D).transpose(1, 2)
        attn = F.softmax((q @ k.transpose(-1, -2)) / math.sqrt(D), dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(out), (k_c, v_c)

In [ ]:
# 验证
torch.manual_seed(0)
gqa = LlamaGQA(256, 8, 2)          # 8 Q 头共享 2 KV 头
x = torch.randn(1, 10, 256)
o, _ = gqa(x); print('GQA out:', o.shape)

mla = MLAAttention(256, 8, 64)     # latent=64 远小于 hidden
o2, cache = mla(x[:, :3])
o3, cache = mla(x[:, 3:4], past_kv=cache)
print('MLA out:', o2.shape, o3.shape, 'cache latent shape:', cache[0].shape)

## 小结 / 易错点
- **GQA** 的共享用 `repeat_interleave(group, dim=head)`，不是 `repeat`（前者按头连续复制，对应"每 group 个 Q 共享 1 个 KV"）。
- **MLA** 的核心收益是 **cache 只存 latent**，长序列 decode 显存大幅下降；代价是多了上采样矩阵。
- RoPE 与低秩压缩冲突：旋转矩阵 $R_m$ 依赖位置，不能和固定上采样矩阵结合，故 DeepSeek 用独立 RoPE 通路绕开。
- 原仓库 MLA 版把 `apply_rotary_pos_emb` 写成类方法却以模块级名字调用会 NameError，本版已修正。